# Notebook 1: Introducción a AWS Bedrock — Invocación de Modelos Base

En este notebook aprenderás a:
- Conectarte a AWS Bedrock usando `boto3`
- Listar los modelos disponibles en tu región
- Invocar un modelo de texto (Claude) con la API `invoke_model`
- Entender la estructura de requests y responses

---

## Prerrequisitos

- Cuenta de AWS con acceso a Bedrock habilitado
- Credenciales configuradas (`aws configure` o variables de entorno)
- Modelos activados en la consola de AWS Bedrock (Model Access)

```bash
pip install boto3
```

---

## 🗺️ El hilo conductor: qué vas a construir en estos cuatro notebooks

Antes de entrar en código, tómate dos minutos para entender el viaje completo. No es solo "cuatro notebooks sueltos". Son cuatro piezas de un mismo puzzle que terminan ensamblando una aplicación de IA real.

---

### El problema que resuelven estos notebooks

Imagina que tu empresa tiene cientos de documentos internos: políticas de RRHH, manuales técnicos, contratos. Un empleado te pregunta: *"¿Cuántos días de vacaciones me corresponden?"*

Un modelo de IA genérico no sabe nada de tus documentos. Puede inventarse una respuesta (lo que se llama "alucinar"). Y si intentas meterle todos los documentos de golpe, el modelo tiene un límite de cuánto puede leer a la vez.

**La solución a este problema se llama RAG** (Retrieval-Augmented Generation, o Generación Aumentada por Recuperación). Y estos cuatro notebooks te enseñan a construirla de cero.

---

### Las cuatro piezas, en orden

**Notebook 1 → La conexión** *(este)*

Aquí aprendes a hablar con los modelos de IA de AWS. Es la fontanería básica: cómo conectarte, cómo enviar una pregunta, cómo leer la respuesta. Sin esto, nada más funciona.

Piénsalo como aprender a marcar un número de teléfono antes de poder mantener una conversación.

---

**Notebook 2 → El idioma**

Los modelos de IA son potentes, pero la calidad de la respuesta depende brutalmente de cómo les preguntas. Aquí aprendes *prompt engineering*: el arte de formular preguntas y dar instrucciones para obtener respuestas útiles, concretas y en el formato que necesitas.

Piénsalo como aprender a dar instrucciones claras a un empleado muy capaz pero muy literal.

---

**Notebook 3 → La memoria**

Aquí entra el concepto más importante de la serie: los **embeddings**. Un embedding convierte cualquier texto en una lista de números que captura su *significado*. Dos frases que dicen lo mismo de forma diferente tendrán números parecidos. Dos frases sin relación tendrán números completamente distintos.

Esto permite buscar documentos por *significado*, no por palabras exactas. Si alguien pregunta "¿cómo proteger mi red?", el sistema encuentra documentos sobre "seguridad informática" aunque no use esas palabras.

Piénsalo como convertir todos tus documentos en puntos en un mapa, donde los documentos relacionados quedan cerca unos de otros.

---

**Notebook 4 → El sistema completo (RAG)**

Aquí juntas todo. Cuando llega una pregunta:
1. La conviertes en embedding y buscas los documentos más relevantes en el mapa
2. Metes esos documentos como contexto en el prompt
3. El modelo responde basándose en esa información real, no en suposiciones

El resultado: un asistente que responde con datos de *tus* documentos, puede decir "no sé" cuando no tiene información, y no inventa nada.

---

### Por qué este orden importa

No puedes construir RAG sin entender embeddings. No puedes sacar partido a los embeddings sin saber formular buenos prompts. Y nada funciona sin saber conectarte a los modelos.

Cada notebook es el suelo sobre el que se asienta el siguiente.

---

Ahora sí. Vamos al código.

## 1. Instalación de dependencias

Lo primero es instalar `boto3`, que es la librería oficial de Python para comunicarse con cualquier servicio de AWS. Sin ella, no hay forma de hablar con Bedrock desde tu código.

In [ ]:
# !pip install boto3 --quiet

## 2. Cargar las credenciales de AWS

Para que tu código pueda acceder a AWS, necesita identificarse: quién eres y qué permisos tienes. Esas credenciales no se escriben directamente en el código (eso sería un riesgo de seguridad); se guardan en un archivo `.env` y se cargan con `dotenv`.

Las variables que se cargan son:
- `AWS_ACCESS_KEY_ID` y `AWS_SECRET_ACCESS_KEY`: son tu "usuario y contraseña" de AWS
- `AWS_DEFAULT_REGION`: la zona geográfica donde está configurado Bedrock (por ejemplo, `eu-west-1` para Irlanda)
- `AWS_BEARER_TOKEN_BEDROCK`: token de acceso adicional para algunos entornos

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
AWS_BEARER_TOKEN_BEDROCK=os.getenv("AWS_BEARER_TOKEN_BEDROCK")
AWS_ACCESS_KEY_ID=os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY=os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION=os.getenv("AWS_DEFAULT_REGION")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

## 3. Importaciones y configuración del cliente

AWS Bedrock expone dos clientes principales en `boto3`:
- **`bedrock`**: para operaciones de gestión (listar modelos, consultar info)
- **`bedrock-runtime`**: para invocar modelos y obtener respuestas

### ¿Qué es un "cliente"?

Un cliente es el objeto en Python que representa la conexión con un servicio de AWS. Cuando lo creas, le dices *a qué servicio quieres conectarte* y *en qué región*. A partir de ahí, usas ese objeto para hacer llamadas al servicio.

Es como crear un "canal de comunicación" con AWS. Una vez creado, puedes usarlo cuantas veces quieras sin volver a configurarlo.

| Cliente | Para qué sirve |
|--------|----------------|
| `bedrock` | Operaciones de gestión: listar modelos disponibles, consultar información |
| `bedrock-runtime` | Invocar modelos: enviar un prompt y recibir una respuesta |

En este notebook usaremos los dos.

In [ ]:
import boto3
import json

# Región donde tienes Bedrock habilitado
REGION = AWS_DEFAULT_REGION

# Cliente para gestión (listar modelos, etc.)
bedrock_client = boto3.client(
    service_name="bedrock",
    region_name=REGION
)

# Cliente para invocar modelos
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

print("✅ Clientes de Bedrock creados correctamente")

## 4. Listar modelos disponibles

Antes de invocar un modelo, puedes consultar qué modelos tienes disponibles en tu cuenta y región. No todos los modelos están habilitados por defecto: hay que activarlos manualmente desde la consola de AWS.

### ¿Qué entra y qué sale?

**Entra:** nada, solo la llamada a `list_foundation_models()`  
**Sale:** una lista con información de cada modelo disponible: su ID, el proveedor (Anthropic, Amazon, Meta...) y qué tipo de datos acepta como entrada (texto, imagen, etc.)

El resultado se usa para saber exactamente qué ID usar al invocar un modelo.

In [ ]:
# Listar todos los modelos base disponibles
response = bedrock_client.list_foundation_models()
modelos = response["modelSummaries"]

print(f"Total de modelos disponibles: {len(modelos)}\n")
print(f"{'Proveedor':<15} {'Model ID':<50} {'Modalidades'}")
print("-" * 90)

for modelo in modelos:
    provider = modelo.get("providerName", "N/A")
    model_id = modelo.get("modelId", "N/A")
    modalities = ", ".join(modelo.get("inputModalities", []))
    print(f"{provider:<15} {model_id:<50} {modalities}")

### Filtrar solo modelos de Anthropic

La lista completa puede ser larga. Este bloque la filtra para quedarse solo con los modelos de Anthropic (los que hacen Claude) que aceptan texto como entrada.

**Entra:** la lista completa de modelos  
**Sale:** una lista reducida, solo con los de Anthropic

In [ ]:
# Filtrar solo modelos de texto de Anthropic
modelos_anthropic = [
    m for m in modelos
    if m.get("providerName") == "Anthropic"
    and "TEXT" in m.get("inputModalities", [])
]

print("Modelos de Anthropic disponibles:")
for m in modelos_anthropic:
    print(f"  - {m['modelId']}")

## 5. Tu primera invocación: Hola Bedrock

Aquí enviamos el primer mensaje real a un modelo de IA. El modelo que se usa es **Claude Haiku**, la versión más ligera y rápida de Claude.

### Cómo funciona una invocación

El proceso tiene tres partes:

1. **Construir el body**: es el "sobre" con tu mensaje. Cada proveedor tiene su propio formato. Para Claude (Anthropic) se usa el formato "Messages API", que es el mismo que usa la API directa de Anthropic.

2. **Invocar el modelo**: se envía ese "sobre" a AWS Bedrock, que lo reenvía al modelo.

3. **Leer la respuesta**: el modelo devuelve un objeto JSON que hay que desempaquetar para leer el texto generado.

### Los campos del body de Claude

| Campo | Qué hace |
|-------|----------|
| `anthropic_version` | Versión del protocolo de comunicación. Siempre `"bedrock-2023-05-31"` para Bedrock |
| `max_tokens` | Máximo de palabras (aproximado) que puede generar el modelo en su respuesta |
| `messages` | La conversación: lista de mensajes con `role` (quién habla) y `content` (qué dice) |

**Entra:** un mensaje de texto  
**Sale:** la respuesta del modelo en formato JSON, de la que extraemos el texto

In [ ]:
# ID del modelo a usar
MODEL_ID = "eu.anthropic.claude-haiku-4-5-20251001-v1:0"

# Construcción del body de la petición (formato Anthropic Messages API)
body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 512,
    "messages": [{"role": "user", "content": "Hello, Claude"}]
}

# Invocar el modelo
response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json"
)

# Decodificar la respuesta
response_body = json.loads(response["body"].read())

print("=== RESPUESTA DEL MODELO ===")
print(response_body["content"][0]["text"])

## 6. Explorar la estructura completa de la respuesta

La respuesta no es solo el texto. Viene acompañada de metadatos que en un entorno real son muy útiles, especialmente para controlar costes y entender por qué el modelo paró de responder.

### El objeto de respuesta completo

Al imprimir la respuesta completa verás una estructura JSON con varias secciones. En el siguiente bloque extraemos las más relevantes.

In [ ]:
print("Estructura completa de la respuesta:")
print(json.dumps(response_body, indent=2, ensure_ascii=False))

### Extraer los campos más importantes

| Campo | Qué significa |
|-------|---------------|
| `content[0]["text"]` | El texto que generó el modelo |
| `usage.input_tokens` | Cuántos tokens (≈ palabras) tenía tu pregunta |
| `usage.output_tokens` | Cuántos tokens generó el modelo en su respuesta |
| `stop_reason` | Por qué paró: `"end_turn"` = terminó por sí solo, `"max_tokens"` = llegó al límite |

Los tokens son la unidad de coste: pagas por los tokens de entrada y los de salida. Saber cuántos usas es clave para estimar el coste de tu aplicación.

In [ ]:
# Extraer información relevante
texto_respuesta = response_body["content"][0]["text"]
tokens_entrada   = response_body["usage"]["input_tokens"]
tokens_salida    = response_body["usage"]["output_tokens"]
stop_reason      = response_body["stop_reason"]

print(f"📝 Texto generado:\n{texto_respuesta}")
print(f"\n📊 Tokens de entrada  : {tokens_entrada}")
print(f"📊 Tokens de salida   : {tokens_salida}")
print(f"🛑 Motivo de parada   : {stop_reason}")

## 7. Ajustar parámetros de inferencia

Hasta ahora hemos enviado prompts con la configuración por defecto. Pero el comportamiento del modelo se puede ajustar con parámetros. Los más importantes:

| Parámetro | Descripción | Rango |
|-----------|-------------|-------|
| `max_tokens` | Máximo de tokens a generar | 1 - 4096 |
| `temperature` | Aleatoriedad de la respuesta | 0.0 - 1.0 |
| `top_p` | Nucleus sampling | 0.0 - 1.0 |
| `stop_sequences` | Secuencias que detienen la generación | Lista de strings |

### ¿Qué es `temperature`?

Es el parámetro más importante para el día a día.

- **`temperature = 0.0`**: el modelo siempre elige la palabra más probable. Las respuestas son repetibles y predecibles. Útil para clasificación, extracción de datos, código.
- **`temperature = 1.0`**: el modelo introduce aleatoriedad. Cada respuesta puede ser diferente. Útil para creatividad, brainstorming.

En el siguiente bloque creamos una función reutilizable `invocar_claude` para no repetir código, y comparamos el efecto de temperature.

In [ ]:
def invocar_claude(prompt, temperature=0.7, max_tokens=512, model_id=MODEL_ID):
    """Función auxiliar para invocar Claude en Bedrock."""
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )
    resultado = json.loads(response["body"].read())
    return resultado["content"][0]["text"]


# Comparar temperatura baja vs alta
prompt_test = "Escribe una frase creativa sobre la inteligencia artificial."

print("🧊 Temperature = 0.0 (determinista):")
print(invocar_claude(prompt_test, temperature=0.0))

print("\n🔥 Temperature = 1.0 (más creativo):")
print(invocar_claude(prompt_test, temperature=1.0))

## 8. Conversaciones multi-turno

Hasta ahora cada llamada es independiente: el modelo no recuerda lo que le dijiste antes. Para simular una conversación real con memoria, hay que pasar el **historial completo** en cada llamada.

### Cómo funciona el historial

La API de Claude acepta una lista de mensajes con roles alternados:
- `"user"`: lo que dice el usuario
- `"assistant"`: lo que respondió el modelo

Cada vez que el usuario hace una nueva pregunta, se añade al historial y se envía todo junto. El modelo "ve" toda la conversación y puede responder coherentemente.

**Entra:** el historial acumulado + el nuevo mensaje del usuario  
**Sale:** la respuesta del modelo, que se añade al historial para la siguiente vuelta

In [ ]:
def chat_bedrock(historial, nuevo_mensaje, model_id=MODEL_ID):
    """Simula una conversación multi-turno con Bedrock."""
    historial.append({"role": "user", "content": nuevo_mensaje})

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "messages": historial
    }
    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )
    resultado = json.loads(response["body"].read())
    respuesta_texto = resultado["content"][0]["text"]

    historial.append({"role": "assistant", "content": respuesta_texto})
    return respuesta_texto, historial


# Iniciar conversación
historial = []

resp1, historial = chat_bedrock(historial, "Hola, me llamo Carlos y estoy aprendiendo Bedrock.")
print(f"Turno 1 - Asistente: {resp1}\n")

resp2, historial = chat_bedrock(historial, "¿Recuerdas cómo me llamo?")
print(f"Turno 2 - Asistente: {resp2}\n")

print(f"Total de mensajes en el historial: {len(historial)}")

## 9. Invocar Amazon Titan (otro proveedor)

Bedrock no es solo Claude. Agrupa modelos de varios proveedores bajo una misma API. Pero ojo: **cada proveedor tiene su propio formato de body**. No es el mismo JSON para Claude que para Amazon Titan o Meta Llama.

### Diferencias de formato

| Proveedor | Campo del texto | Campo de configuración |
|-----------|----------------|----------------------|
| Anthropic (Claude) | `messages[].content` | `max_tokens`, `temperature` |
| Amazon Titan | `inputText` | `textGenerationConfig.maxTokenCount` |
| Meta Llama | `prompt` | `max_gen_len` |

**Entra:** un texto en el campo `inputText` (formato Titan)  
**Sale:** la respuesta en `results[0]["outputText"]` (formato Titan, diferente a Claude)

In [ ]:
TITAN_MODEL_ID = "amazon.titan-text-express-v1"

# Formato específico de Amazon Titan
body_titan = {
    "inputText": "Explica en una oración qué es el machine learning.",
    "textGenerationConfig": {
        "maxTokenCount": 256,
        "temperature": 0.5,
        "topP": 0.9
    }
}

response_titan = bedrock_runtime.invoke_model(
    modelId=TITAN_MODEL_ID,
    body=json.dumps(body_titan),
    contentType="application/json",
    accept="application/json"
)

resultado_titan = json.loads(response_titan["body"].read())
print("Respuesta de Amazon Titan:")
print(resultado_titan["results"][0]["outputText"])

## 10. Resumen y próximos pasos

En este notebook aprendiste a:

✅ Crear clientes `bedrock` y `bedrock-runtime` con boto3  
✅ Listar y filtrar modelos disponibles  
✅ Invocar Claude con la Messages API  
✅ Interpretar la estructura de la respuesta (tokens, stop_reason)  
✅ Ajustar parámetros como `temperature` y `max_tokens`  
✅ Mantener conversaciones multi-turno  
✅ Invocar modelos de distintos proveedores (Anthropic vs Amazon)  

**Siguiente notebook →** Prompt Engineering aplicado a Bedrock